In [1]:
%load_ext aiida
%aiida

Loaded AiiDA DB environment - profile name: bit.

In [2]:
from aiida import orm
from monty.serialization import loadfn, dumpfn
import numpy as np
from aiida_vasp.workchains import VaspHybridBandsWorkChain
from aiida_vasp.workchains.v2 import VaspHybridBandUpdater, VaspRelaxUpdater
from aiida_grouppathx import GroupPathX
from tqdm import tqdm

In [3]:
# Load the structures to be calculated
dataset = loadfn('binary_mbj_calc_structures_2025_03_19.json')
print(len(dataset))

665


In [6]:
basepath = GroupPathX('hc-binary-mbj')
structpath = basepath['structures']

Deposit the structures

Sort the nodes based on their number of sites

In [7]:
nodes = [x.get_node() for x in structpath]
nodes.sort(key=lambda x: len(x.sites))

In [8]:
nodes[-1].get_ase()

Atoms(symbols='Ru4F24', pbc=True, cell=[[0.0, 0.0, -4.82221923], [0.0, -8.38575766, 0.0], [-9.05755894, 0.0, 0.0]], masses=...)

## Setting up the calculation

In [28]:
def callback(node, label):
    """Generate process builder"""
    #queue_name = 'tyhcnormal'
    queue_name = 'xhhctdnormal'
    code = 'vasp-6.4.2@sugon-xh-v2'
    upd = VaspHybridBandUpdater().apply_preset(structure=node, overrides={
        'lhfcalc': True,
        'hfscreen': 0.2,
        'precfock': 'fast',
        'ispin': 2,
        'gga': None,  # PBE functional
        'magmom': None,
        'ncore': 2,
        'kpar': 8,
    },
        # code='vasp-6.4.2@sugon-tai',
        code=code,
        label=f'{node.get_formula()} {node.label} MP STRUCT HSE06 NORELAX')
    upd.set_resources(num_machines=1, tot_num_mpiprocs=64)
    upd.set_options(max_wallclock_seconds=3600 * 48, queue_name=queue_name)
    
    # Reuse WAVECAR for the mbj band structure calculation
    
    #upd.set_band_settings(hybrid_reuse_wavecar=True)
    
    # Only re-try twice if there is convergence problem
    #upd.builder.scf.max_iterations = 2

    # Do relaxation - this is needed for getting the IBZKPT
    upd_relax = VaspRelaxUpdater(builder=upd.builder.relax)
    upd_relax.apply_preset(structure=node, overrides={
        #'metagga': 'mbj',
        'ispin': 2,
        'gga': None,
        'magmom': None,
        'ncore': 8,
    },
         code=code, label=f'{node.get_formula()} {node.label} MP STRUCT SP')
    upd_relax.set_resources(num_machines=1, tot_num_mpiprocs=32)
    upd_relax.set_options(max_wallclock_seconds=3600 * 12, queue_name=queue_name)
    upd_relax.set_relax_settings(perform=False) # Not actually relaxing
    
    upd.set_band_settings(band_mode='bradcrack', line_density=10, kpoints_per_split=200)
    running = upd.submit()
    return running, label

In [29]:
workpath = basepath['hse06_bandstructure_works']
workpath.get_or_create_group()

(<Group: 'hc-binary-mbj/hse06_bandstructure_works' [type core], of user bzhu@bit.edu.cn>,
 False)

In [30]:
from aiida_grouppathx.launch_manager import GroupLauncher

In [31]:
launcher = GroupLauncher(workpath, 3, callback, source_key_obj_pairs=[(node.label, node) for node in nodes], logfile='launch_2025_4_6.log')

In [32]:
launcher.launch_loop()

Total number of running jobs: 2
Total number of jobs to run : 659
Time elapsed to gather jobs: 0.02 seconds
Slot usage: 2/3
Launching 1 jobs...
Launched 1 jobs...
Total number of running jobs: 3
Total number of jobs to run : 658
Time elapsed to gather jobs: 0.03 seconds
Slot usage: 3/3
Total number of running jobs: 3
Total number of jobs to run : 658
Time elapsed to gather jobs: 0.03 seconds
Slot usage: 3/3
Total number of running jobs: 3
Total number of jobs to run : 658
Time elapsed to gather jobs: 0.04 seconds
Slot usage: 3/3
Total number of running jobs: 3
Total number of jobs to run : 658
Time elapsed to gather jobs: 0.04 seconds
Slot usage: 3/3
Total number of running jobs: 3
Total number of jobs to run : 658
Time elapsed to gather jobs: 0.04 seconds
Slot usage: 3/3
Total number of running jobs: 3
Total number of jobs to run : 658
Time elapsed to gather jobs: 0.03 seconds
Slot usage: 3/3
Total number of running jobs: 3
Total number of jobs to run : 658
Time elapsed to gather jobs

KeyboardInterrupt: 